# データサイエンス特論 第14回 演習課題
青山学院大学大学院 理工学研究科 理工学専攻 知能情報コース 修士1年 35626302 森下剛

In [61]:
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, mean_squared_error, root_mean_squared_error

## k最近傍法

#### 演習問題1の解答

In [62]:
# 乳がんデータセットの読み込み
data = load_breast_cancer()
X, y = data.data, data.target

In [63]:
# 全体の20%をテストデータとして分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

# kを変化させた場合の検証
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"k = {k} : accuracy = {acc}")


k = 1 : accuracy = 0.9210526315789473
k = 2 : accuracy = 0.9210526315789473
k = 3 : accuracy = 0.9210526315789473
k = 4 : accuracy = 0.9035087719298246
k = 5 : accuracy = 0.9385964912280702
k = 6 : accuracy = 0.9473684210526315
k = 7 : accuracy = 0.9385964912280702
k = 8 : accuracy = 0.9473684210526315
k = 9 : accuracy = 0.9298245614035088
k = 10 : accuracy = 0.9298245614035088
k = 11 : accuracy = 0.9298245614035088
k = 12 : accuracy = 0.9298245614035088
k = 13 : accuracy = 0.9210526315789473
k = 14 : accuracy = 0.9210526315789473
k = 15 : accuracy = 0.9210526315789473
k = 16 : accuracy = 0.9210526315789473
k = 17 : accuracy = 0.9122807017543859
k = 18 : accuracy = 0.9210526315789473
k = 19 : accuracy = 0.9122807017543859
k = 20 : accuracy = 0.9122807017543859


#### 演習問題2の解答

In [64]:
# 標準化
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# kを変化させた場合の検証
print("k    before    after")
for k in range(1, 21):
    # 標準化前
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc_raw = accuracy_score(y_test, knn.predict(X_test))

    # 標準化後
    knn_std = KNeighborsClassifier(n_neighbors=k)
    knn_std.fit(X_train_std, y_train)
    acc_std = accuracy_score(y_test, knn_std.predict(X_test_std))

    print(f"{k}   {acc_raw}    {acc_std}")


k    before    after
1   0.9210526315789473    0.9385964912280702
2   0.9210526315789473    0.956140350877193
3   0.9210526315789473    0.9473684210526315
4   0.9035087719298246    0.9298245614035088
5   0.9385964912280702    0.956140350877193
6   0.9473684210526315    0.9473684210526315
7   0.9385964912280702    0.956140350877193
8   0.9473684210526315    0.956140350877193
9   0.9298245614035088    0.956140350877193
10   0.9298245614035088    0.956140350877193
11   0.9298245614035088    0.9473684210526315
12   0.9298245614035088    0.956140350877193
13   0.9210526315789473    0.956140350877193
14   0.9210526315789473    0.9473684210526315
15   0.9210526315789473    0.956140350877193
16   0.9210526315789473    0.9473684210526315
17   0.9122807017543859    0.956140350877193
18   0.9210526315789473    0.956140350877193
19   0.9122807017543859    0.956140350877193
20   0.9122807017543859    0.9473684210526315


標準化後の精度向上が確認できた。

#### 演習問題3の解答

In [65]:
# 糖尿病データセットの読み込み
data = load_diabetes()
X, y = data.data, data.target

In [66]:
# 全体の20%をテストデータとして分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

# k最近傍回帰
knr = KNeighborsRegressor()
knr.fit(X_train, y_train)
rmse_knr = root_mean_squared_error(y_test, knr.predict(X_test))
print(f"k最近傍回帰の RMSE: {rmse_knr}")

# 重回帰との比較
lr = LinearRegression()
lr.fit(X_train, y_train)
rmse_lr = root_mean_squared_error(y_test, lr.predict(X_test))
print(f"重回帰の RMSE: {rmse_lr}")


k最近傍回帰の RMSE: 62.90614374882881
重回帰の RMSE: 54.70449002870804


重回帰のほうが予測精度が高かった。糖尿病データの次元は多いことからk最近傍回帰の精度が悪化したと考えられる。

#### 演習問題4の解答

In [67]:
knr_dist = KNeighborsRegressor(weights='distance')
knr_dist.fit(X_train, y_train)
rmse_dist = root_mean_squared_error(y_test, knr_dist.predict(X_test))
print(f"weights='distance' の RMSE: {rmse_dist}")

weights='distance' の RMSE: 62.50595261162572


#### 演習問題5の解答

In [68]:
# 探索するパラメータの範囲
param_grid = {
    'n_neighbors': range(1, 21),
    'weights': ['uniform', 'distance'],
}

# グリッドサーチ
gs = GridSearchCV(KNeighborsRegressor(), param_grid, cv=10,scoring='neg_mean_squared_error')
gs.fit(X_train, y_train)

print(f"最適なパラメータ: {gs.best_params_}")

# 最適なパラメータで平均二乗誤差を計算
y_pred = gs.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse}")


最適なパラメータ: {'n_neighbors': 16, 'weights': 'uniform'}
MSE: 3357.0829968398875


## パイプライン

#### 演習問題6の解答

In [69]:
# 乳がんデータセットの読み込み
data = load_breast_cancer()
X, y = data.data, data.target

In [70]:
# デフォルト設定の k 最近傍法を10分割交差検証で評価
knn = KNeighborsClassifier()
scores = cross_val_score(knn, X, y, cv=10, scoring='accuracy')

print(f"各分割の accuracy: {scores}")
print(f"平均 accuracy: {scores.mean()}")


各分割の accuracy: [0.9122807  0.87719298 0.89473684 0.96491228 0.94736842 0.92982456
 0.96491228 0.92982456 0.9122807  0.96428571]
平均 accuracy: 0.9297619047619046


#### 演習問題7の解答

In [71]:
# パイプラインを構成
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())

# 10分割交差検証で評価
scores = cross_val_score(pipe, X, y, cv=10, scoring='accuracy')

print(f"各分割の accuracy: {scores}")
print(f"平均 accuracy: {scores.mean()}")


各分割の accuracy: [0.98245614 0.96491228 0.92982456 0.98245614 1.         0.96491228
 0.94736842 0.96491228 0.94736842 0.98214286]
平均 accuracy: 0.9666353383458647


#### 演習問題8の解答

In [72]:
# パイプラインを構成
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())

# 探索するパラメータの範囲
param_grid = {
    'kneighborsclassifier__n_neighbors': range(1, 21),
    'kneighborsclassifier__weights': ['uniform', 'distance'],
}

# 最適なパラメータを探索
gs = GridSearchCV(pipe, param_grid, cv=10, scoring='accuracy')

# 全体を10分割交差検証で評価
scores = cross_val_score(gs, X, y, cv=10, scoring='accuracy')

print(f"各分割の accuracy: {scores}")
print(f"平均 accuracy: {scores.mean()}")


各分割の accuracy: [0.98245614 0.96491228 0.92982456 0.98245614 1.         0.98245614
 0.92982456 0.96491228 0.96491228 0.96428571]
平均 accuracy: 0.9666040100250626
